In [1]:
import os, csv, math, time, warnings
from datetime import datetime

import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.options.mode.copy_on_write = True
warnings.filterwarnings("ignore")

load_dotenv()
alpha_vantage_api_key = os.getenv("alpha_vantage_api_key")
os.chdir(r'D:\Investment')


# Market Data


In [2]:
def get_forex_df(from_currency, to_currency, api_key):
    pair = f"{from_currency}{to_currency}"
    url  = (f"https://www.alphavantage.co/query?function=FX_MONTHLY"
            f"&from_symbol={from_currency}&to_symbol={to_currency}&apikey={api_key}")
    data = requests.get(url, timeout=30).json()
    key  = "Time Series FX (Monthly)"
    if key not in data:
        raise ValueError(f"No FX data: {data}")
    monthly_df = (
        pd.DataFrame(data[key]).transpose()[["4. close"]]
        .rename(columns={"4. close": pair})
    )
    monthly_df[pair]  = pd.to_numeric(monthly_df[pair], errors="coerce")
    monthly_df.index  = pd.to_datetime(monthly_df.index)
    monthly_df["Year"]  = monthly_df.index.year
    monthly_df["Month"] = monthly_df.index.month
    yearly_df = monthly_df.groupby("Year")[pair].mean().to_frame(name=pair)
    yearly_df["Year"] = yearly_df.index
    return monthly_df.sort_index(ascending=False), yearly_df.sort_index(ascending=False)


fx_CADUSD_monthly_df, fx_CADUSD_yrly_df = get_forex_df("CAD", "USD", alpha_vantage_api_key)
print("FX monthly:", fx_CADUSD_monthly_df.index[0].date(), "->", fx_CADUSD_monthly_df.index[-1].date())

_fx_monthly_lookup = {
    (int(r["Year"]), int(r["Month"])): float(r["CADUSD"])
    for _, r in fx_CADUSD_monthly_df.iterrows()
}

def _cadusd_for_date(date):
    '''CADUSD rate for the year-month of `date`. Falls back to yearly avg if month missing.'''
    return _fx_monthly_lookup.get(
        (int(date.year), int(date.month)),
        float(fx_CADUSD_yrly_df.loc[int(date.year), "CADUSD"]),
    )


FX monthly: 2026-05-01 -> 2010-10-29


In [3]:
url  = (f"https://www.alphavantage.co/query?function=TIME_SERIES_DAILY"
        f"&symbol=SPY&apikey={alpha_vantage_api_key}&outputsize=full")
data = requests.get(url, timeout=30).json()
if "Time Series (Daily)" not in data:
    raise ValueError(f"No SPY data: {data}")

sp500_Daily_stock_df = (
    pd.DataFrame(data["Time Series (Daily)"]).transpose()[["4. close"]]
    .rename(columns={"4. close": "stock_price"})
)
sp500_Daily_stock_df["stock_price"] = sp500_Daily_stock_df["stock_price"].astype(float).round(2)
sp500_Daily_stock_df.index = pd.to_datetime(sp500_Daily_stock_df.index)

sp500_hist_annual = sp500_Daily_stock_df.resample("Y").last()
sp500_hist_annual.index = sp500_hist_annual.index.year
sp500_hist_annual["Year"] = sp500_hist_annual.index
sp500_hist_annual["Annual Return"] = sp500_hist_annual["stock_price"].pct_change() * 100
sp500_hist_annual = sp500_hist_annual.dropna(subset=["Annual Return"]).sort_index(ascending=False)


In [4]:
def _av_get_json(url, max_retries=4):
    backoff = 1.0
    for _ in range(max_retries):
        time.sleep(0.25)
        data = requests.get(url, timeout=30).json()
        info = data.get("Information", "")
        if "Burst pattern" in info or "rate limit" in info.lower():
            print(f"    [throttled] sleeping {backoff:.1f}s")
            time.sleep(backoff); backoff *= 2; continue
        return data
    raise RuntimeError(f"Alpha Vantage burst limit not clearing: {data}")


def fetch_av_daily_split_adjusted(symbol, api_key):
    '''Return (Daily_stock_df, stock_split_record_df) for `symbol`, split-adjusted.'''
    splits = _av_get_json(
        f"https://www.alphavantage.co/query?function=SPLITS&symbol={symbol}&apikey={api_key}"
    ).get("data", [])
    if splits:
        stock_split_record_df = pd.DataFrame(splits)
        stock_split_record_df["split_factor"]   = pd.to_numeric(stock_split_record_df["split_factor"], errors="coerce")
        stock_split_record_df["effective_date"] = pd.to_datetime(stock_split_record_df["effective_date"])
    else:
        stock_split_record_df = pd.DataFrame({"split_factor": [1], "effective_date": [datetime.today()]})

    series = _av_get_json(
        f"https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol={symbol}&apikey={api_key}&outputsize=full"
    ).get("Time Series (Daily)")
    if series is None:
        raise ValueError(f"No daily data for {symbol}")

    Daily_stock_df = (
        pd.DataFrame(series).transpose()[["4. close"]]
        .rename(columns={"4. close": "stock_price"})
    )
    Daily_stock_df["stock_price"] = Daily_stock_df["stock_price"].astype(float).round(2)
    Daily_stock_df.index = pd.to_datetime(Daily_stock_df.index)

    for eff_date in stock_split_record_df["effective_date"].dt.date:
        factor = stock_split_record_df.loc[
            stock_split_record_df["effective_date"].dt.date == eff_date, "split_factor"
        ].values[0]
        Daily_stock_df.loc[Daily_stock_df.index.date < eff_date, "stock_price"] /= factor

    return Daily_stock_df, stock_split_record_df


# Questrade


In [5]:
questrade_files = [
    'questrade_Inv_activity_2019-2021.xlsx',
    'questrade_Inv_activity_2022.xlsx',
    'questrade_Inv_activity_2023.xlsx',
    'questrade_Inv_activity_2024.xlsx',
    'questrade_Inv_activity_2025.xlsx',
    'questrade_Inv_activity_2026.xlsx',
]

inv_activity_df = pd.concat([pd.read_excel(f) for f in questrade_files], ignore_index=True)
inv_activity_df["Transaction Date"] = pd.to_datetime(inv_activity_df["Transaction Date"])
inv_activity_df["Year"] = inv_activity_df["Transaction Date"].dt.year
inv_activity_df = inv_activity_df[inv_activity_df["Year"] >= 2019]
inv_activity_df.rename(columns={
    "Transaction Date": "Transaction_Date",
    "Settlement Date":  "Settlement_Date",
    "Net Amount":       "Net_Amount",
}, inplace=True)


In [6]:
_qt_ddw_categories = {
    "Deposits":    ["CON", "FCH", "DEP"],
    "Withdrawals": ["WDR"],
    "Dividends":   ["DIV", "Dividends"],
}
_yearly_totals_qt = {}
for category, actions in _qt_ddw_categories.items():
    filtered_df = inv_activity_df[
        inv_activity_df["Action"].isin(actions)
        | inv_activity_df["Activity Type"].isin(actions)
    ].copy()
    _cad = filtered_df["Currency"] == "CAD"
    filtered_df.loc[_cad, "Net_Amount"] = filtered_df.loc[_cad].apply(
        lambda row: row["Net_Amount"] * _cadusd_for_date(row["Transaction_Date"]), axis=1
    )
    _yearly_totals_qt[category] = filtered_df.groupby("Year")["Net_Amount"].sum()

inv_yrly_base = pd.DataFrame(_yearly_totals_qt).fillna(0)

_HARDCODED_2019_DEPOSIT_CAD = 6000.0
inv_yrly_base.at[2019, "Deposits"] = (
    _HARDCODED_2019_DEPOSIT_CAD * _cadusd_for_date(pd.Timestamp("2019-01-02"))
)

# Per-transaction cash flows for Modified Dietz / XIRR
_q_cf_actions = ["CON", "FCH", "DEP", "WDR"]
questrade_cf_df = inv_activity_df[
    inv_activity_df["Action"].isin(_q_cf_actions)
    | inv_activity_df["Activity Type"].isin(_q_cf_actions)
][["Transaction_Date", "Net_Amount", "Currency", "Year"]].copy()
_cad = questrade_cf_df["Currency"] == "CAD"
questrade_cf_df.loc[_cad, "Net_Amount"] = questrade_cf_df.loc[_cad].apply(
    lambda row: row["Net_Amount"] * _cadusd_for_date(row["Transaction_Date"]), axis=1
)
questrade_cf_df = questrade_cf_df[["Transaction_Date", "Net_Amount"]].reset_index(drop=True)

# Reconcile 2019 synthetic deposit
_target_2019_usd = _HARDCODED_2019_DEPOSIT_CAD * _cadusd_for_date(pd.Timestamp("2019-01-02"))
_actual_2019_cf  = questrade_cf_df.loc[questrade_cf_df["Transaction_Date"].dt.year == 2019, "Net_Amount"].sum()
_missing_2019    = _target_2019_usd - _actual_2019_cf
if _missing_2019 > 0:
    questrade_cf_df = pd.concat([
        questrade_cf_df,
        pd.DataFrame([{"Transaction_Date": pd.Timestamp("2019-01-02"), "Net_Amount": _missing_2019}]),
    ], ignore_index=True)
    print(f"2019 reconcile: added synthetic ${_missing_2019:,.2f} USD")
else:
    print(f"2019 reconcile: no synthetic CF needed")


2019 reconcile: added synthetic $1,513.36 USD


In [7]:
_exclude_symbols = ['DLR.TO', 'G036247', 'H038778', 'DLR', 'IVV', 'VYM', 'VFH', 'KWEB', 'BYND']
trades_df_questrade = inv_activity_df[
    (inv_activity_df["Activity Type"] == "Trades")
    & (~inv_activity_df["Symbol"].isin(_exclude_symbols))
].copy()

if datetime.today().year not in sorted(trades_df_questrade["Year"].unique()):
    print("No current year Questrade trades — creating synthetic records")
    trades_df_questrade = trades_df_questrade.sort_values("Transaction_Date").reset_index(drop=True)
    last = trades_df_questrade.iloc[-1]
    _cur_yr = datetime.now().year
    buy_row  = pd.DataFrame([last])
    sell_row = pd.DataFrame([last])
    buy_row["Action"]           = "Buy"
    buy_row["Transaction_Date"] = pd.Timestamp(f"{_cur_yr}-01-01")
    buy_row["Settlement_Date"]  = pd.Timestamp(f"{_cur_yr}-01-01")
    buy_row["Year"]             = _cur_yr
    buy_row["Quantity"]         = abs(last["Quantity"])
    buy_row["Net_Amount"]       = abs(last["Quantity"]) * last["Price"] * -1
    sell_row["Action"]           = "Sell"
    sell_row["Transaction_Date"] = pd.Timestamp(f"{_cur_yr}-01-02")
    sell_row["Settlement_Date"]  = pd.Timestamp(f"{_cur_yr}-01-02")
    sell_row["Year"]             = _cur_yr
    sell_row["Quantity"]         = abs(last["Quantity"]) * -1
    sell_row["Net_Amount"]       = abs(last["Quantity"]) * last["Price"]
    trades_df_questrade = pd.concat([trades_df_questrade, buy_row, sell_row], ignore_index=True)

# CAD -> USD conversion using per-transaction monthly FX
_cad = trades_df_questrade["Currency"] == "CAD"
trades_df_questrade.loc[_cad, "Net_Amount"] = trades_df_questrade.loc[_cad].apply(
    lambda row: row["Net_Amount"] * _cadusd_for_date(row["Transaction_Date"]), axis=1
)
trades_df_questrade.loc[_cad, "Price"] = trades_df_questrade.loc[_cad].apply(
    lambda row: row["Price"] * _cadusd_for_date(row["Transaction_Date"]), axis=1
)

trades_df_questrade = trades_df_questrade[[
    "Transaction_Date", "Action", "Symbol", "Quantity", "Price", "Net_Amount", "Year"
]]


In [8]:
open_buy_df_previous = pd.DataFrame()

for i in sorted(trades_df_questrade["Year"].unique()):
    open_stock_mkt_value_list = []
    yearly_trades_df = trades_df_questrade[trades_df_questrade["Year"] == i]
    inv_yrly_base.loc[inv_yrly_base.index == i, "Closed_gain/loss"] = yearly_trades_df["Net_Amount"].sum()

    yearly_trades_df = pd.concat([yearly_trades_df, open_buy_df_previous], ignore_index=True)
    buy_df  = yearly_trades_df[yearly_trades_df["Action"] == "Buy"]
    sell_df = yearly_trades_df[yearly_trades_df["Action"] == "Sell"]

    buy_grp  = buy_df.groupby("Symbol")["Quantity"].sum()
    sell_grp = sell_df.groupby("Symbol")["Quantity"].sum().reindex(buy_grp.index, fill_value=0)
    open_grp_df = (buy_grp + sell_grp)[(buy_grp + sell_grp) > 0].reset_index()

    open_buy_df = pd.merge(open_grp_df, buy_df, on="Symbol", how="left")
    open_buy_df = open_buy_df.drop_duplicates(subset=["Symbol", "Quantity_x"], keep="first")
    open_buy_df = open_buy_df[[
        "Transaction_Date", "Action", "Symbol", "Quantity_x", "Price", "Net_Amount", "Year"
    ]].rename(columns={"Quantity_x": "Quantity"})
    open_buy_df_previous = open_buy_df.copy()

    print(i, open_buy_df["Symbol"].unique())
    for symbol in open_buy_df["Symbol"].unique():
        Daily_stock_df, _ = fetch_av_daily_split_adjusted(symbol, alpha_vantage_api_key)
        yr_df = Daily_stock_df.resample("Y").last()

        if symbol.endswith(".TO"):
            raw_price = yr_df.loc[yr_df.index.year == i, "stock_price"].values[0]
            _dec_fx = _fx_monthly_lookup.get(
                (int(i), 12),
                _fx_monthly_lookup.get((int(i), 11), float(fx_CADUSD_yrly_df.loc[int(i), "CADUSD"]))
            )
            open_stock_price = raw_price * _dec_fx
        else:
            open_stock_price = yr_df.loc[yr_df.index.year == i, "stock_price"].values[0]

        qty = open_buy_df.loc[open_buy_df["Symbol"] == symbol, "Quantity"].values.sum()
        open_stock_mkt_value_list.append(qty * open_stock_price)
        print(f"  {symbol}: qty={qty}, price={open_stock_price:.2f}, value={qty*open_stock_price:.2f}")

    inv_yrly_base.loc[inv_yrly_base.index == i, "Open_mkt_value"] = round(sum(open_stock_mkt_value_list), 2)
    print(f"  Year {i} total open mkt value: {round(sum(open_stock_mkt_value_list),2)}\n")


2019 ['BABA' 'DDOG' 'ORCL']
  BABA: qty=20.0, price=212.10, value=4242.00
  DDOG: qty=9.0, price=37.78, value=340.02
  ORCL: qty=12.0, price=52.98, value=635.76
  Year 2019 total open mkt value: 5217.78

2020 ['AAL' 'AMZN' 'DDOG']
  AAL: qty=20.0, price=15.77, value=315.40
  AMZN: qty=2.0, price=162.85, value=325.69
  DDOG: qty=50.0, price=98.44, value=4922.00
  Year 2020 total open mkt value: 5563.09

2021 ['FDX' 'META' 'TSM']
  FDX: qty=10.0, price=258.64, value=2586.40
  META: qty=20.0, price=336.35, value=6727.00
  TSM: qty=20.0, price=120.31, value=2406.20
  Year 2021 total open mkt value: 11719.6

2022 ['FDX' 'META' 'TSM']
  FDX: qty=10.0, price=173.20, value=1732.00
  META: qty=20.0, price=120.34, value=2406.80
  TSM: qty=20.0, price=74.49, value=1489.80
  Year 2022 total open mkt value: 5628.6

2023 ['TSM']
  TSM: qty=40.0, price=104.00, value=4160.00
  Year 2023 total open mkt value: 4160.0

2024 ['GOOG' 'TGT' 'TSM']
  GOOG: qty=67.0, price=190.44, value=12759.48
  TGT: qty=22

# Moomoo


In [9]:
_moomoo_files = [
    'moomoo_inv_activity_2024.csv',
    'moomoo_inv_activity_2025.csv',
    'moomoo_inv_activity_2026.csv',
]
inv_activity_df_moomoo_raw = pd.concat(
    [pd.read_csv(f) for f in _moomoo_files], ignore_index=True
)
inv_activity_df_moomoo_raw["ReportDate"] = pd.to_datetime(
    inv_activity_df_moomoo_raw["ReportDate"], format="%Y%m%d"
)
inv_activity_df_moomoo_raw["Year"] = inv_activity_df_moomoo_raw["ReportDate"].dt.year


In [10]:
# Filter: (CAD & DEP) | DIV | WITH  — preserves original operator-precedence behaviour
_mm_raw = inv_activity_df_moomoo_raw
_mm_ddw = _mm_raw[
    ((_mm_raw["CurrencyPrimary"] == "CAD") & (_mm_raw["ActivityCode"] == "DEP"))
    | _mm_raw["ActivityCode"].isin(["DIV", "WITH"])
][["ReportDate", "ActivityCode", "Amount", "CurrencyPrimary", "TransactionID", "Year"]].copy()
_mm_ddw = _mm_ddw.sort_values("ReportDate").drop_duplicates(["TransactionID"])

_mm_ddw_categories = {
    "Deposits":    ["DEP"],
    "Withdrawals": ["WITH"],
    "Dividends":   ["DIV"],
}
_yearly_totals_mm = {}
for category, actions in _mm_ddw_categories.items():
    filtered = _mm_ddw[_mm_ddw["ActivityCode"].isin(actions)].copy()
    _cad = filtered["CurrencyPrimary"] == "CAD"
    filtered.loc[_cad, "Amount"] = filtered.loc[_cad].apply(
        lambda row: row["Amount"] * _cadusd_for_date(row["ReportDate"]), axis=1
    )
    _yearly_totals_mm[category] = filtered.groupby("Year")["Amount"].sum()

inv_yrly_base_moomoo = pd.DataFrame(_yearly_totals_mm).fillna(0)

# Cash flows (deposits + withdrawals only; dividends are internal)
moomoo_cf_df = _mm_ddw[_mm_ddw["ActivityCode"].isin(["DEP", "WITH"])][
    ["ReportDate", "Amount", "CurrencyPrimary"]
].copy()
_cad = moomoo_cf_df["CurrencyPrimary"] == "CAD"
moomoo_cf_df.loc[_cad, "Amount"] = moomoo_cf_df.loc[_cad].apply(
    lambda row: row["Amount"] * _cadusd_for_date(row["ReportDate"]), axis=1
)
moomoo_cf_df = (
    moomoo_cf_df.rename(columns={"ReportDate": "Transaction_Date", "Amount": "Net_Amount"})
    [["Transaction_Date", "Net_Amount"]].reset_index(drop=True)
)


In [11]:
trades_df_moomoo = _mm_raw[
    _mm_raw["ActivityCode"].isin(["BUY", "SELL"])
    & _mm_raw["AssetClass"].isin(["CASH", "STK"])
    & (_mm_raw["CurrencyPrimary"] == "USD")
][["ReportDate", "ActivityCode", "Buy/Sell", "TradeQuantity", "TradeGross",
   "Amount", "CurrencyPrimary", "AssetClass", "Symbol", "Year"]].copy()

if datetime.today().year not in sorted(trades_df_moomoo["Year"].unique()):
    print("No current year Moomoo trades — creating synthetic records")
    trades_df_moomoo = trades_df_moomoo.sort_values("ReportDate").reset_index(drop=True)
    last = trades_df_moomoo.iloc[-1]
    _cur_yr = datetime.now().year
    buy_row = pd.DataFrame([last]); sell_row = pd.DataFrame([last])
    buy_row["ActivityCode"] = "BUY";  buy_row["Buy/Sell"] = "Buy"
    buy_row["ReportDate"]   = pd.Timestamp(f"{_cur_yr}-01-01"); buy_row["Year"] = _cur_yr
    buy_row["TradeQuantity"] = abs(last["TradeQuantity"])
    buy_row["Amount"]        = abs(last["Amount"]) * -1
    sell_row["ActivityCode"] = "SELL"; sell_row["Buy/Sell"] = "Sell"
    sell_row["ReportDate"]   = pd.Timestamp(f"{_cur_yr}-01-02"); sell_row["Year"] = _cur_yr
    sell_row["TradeQuantity"] = abs(last["TradeQuantity"]) * -1
    sell_row["Amount"]        = abs(last["Amount"])
    trades_df_moomoo = pd.concat([trades_df_moomoo, buy_row, sell_row], ignore_index=True)

trades_df_moomoo = trades_df_moomoo.rename(columns={
    "ReportDate":    "Transaction_Date",
    "ActivityCode":  "Action",
    "TradeQuantity": "Quantity",
    "Amount":        "Net_Amount",
})
trades_df_moomoo["Action"] = trades_df_moomoo["Action"].replace({"BUY": "Buy", "SELL": "Sell"})
trades_df_moomoo["Price"]  = trades_df_moomoo["TradeGross"] / trades_df_moomoo["Quantity"]
trades_df_moomoo = trades_df_moomoo[[
    "Transaction_Date", "Action", "Symbol", "Quantity", "Price", "Net_Amount", "Year"
]]


In [12]:
open_buy_df_moomoo_previous = pd.DataFrame()

for i in sorted(trades_df_moomoo["Year"].unique()):
    open_stock_mkt_value_moomoo_list = []
    yearly_trades_df_moomoo = trades_df_moomoo[trades_df_moomoo["Year"] == i]
    inv_yrly_base_moomoo.loc[inv_yrly_base_moomoo.index == i, "Closed_gain/loss"] = (
        yearly_trades_df_moomoo["Net_Amount"].sum()
    )

    yearly_trades_df_moomoo = pd.concat(
        [yearly_trades_df_moomoo, open_buy_df_moomoo_previous], ignore_index=True
    )
    buy_df_mm  = yearly_trades_df_moomoo[yearly_trades_df_moomoo["Action"] == "Buy"]
    sell_df_mm = yearly_trades_df_moomoo[yearly_trades_df_moomoo["Action"] == "Sell"]

    buy_grp_mm  = buy_df_mm.groupby("Symbol")["Quantity"].sum()
    sell_grp_mm = sell_df_mm.groupby("Symbol")["Quantity"].sum().reindex(buy_grp_mm.index, fill_value=0)
    open_grp_mm_df = (buy_grp_mm + sell_grp_mm)[(buy_grp_mm + sell_grp_mm) > 0].reset_index()

    open_buy_df_moomoo = pd.merge(open_grp_mm_df, buy_df_mm, on="Symbol", how="left")
    open_buy_df_moomoo = open_buy_df_moomoo.drop_duplicates(subset=["Symbol", "Quantity_x"], keep="first")
    open_buy_df_moomoo = open_buy_df_moomoo[[
        "Transaction_Date", "Action", "Symbol", "Quantity_x", "Price", "Net_Amount", "Year"
    ]].rename(columns={"Quantity_x": "Quantity"})
    open_buy_df_moomoo_previous = open_buy_df_moomoo.copy()

    print(i, open_buy_df_moomoo["Symbol"].unique())
    for symbol in open_buy_df_moomoo["Symbol"].unique():
        Daily_stock_df, _ = fetch_av_daily_split_adjusted(symbol, alpha_vantage_api_key)
        yr_df = Daily_stock_df.resample("Y").last()
        open_stock_price = yr_df.loc[yr_df.index.year == i, "stock_price"].values[0]
        qty = open_buy_df_moomoo.loc[open_buy_df_moomoo["Symbol"] == symbol, "Quantity"].values.sum()
        open_stock_mkt_value_moomoo_list.append(qty * open_stock_price)
        print(f"  {symbol}: qty={qty}, price={open_stock_price:.2f}, value={qty*open_stock_price:.2f}")

    inv_yrly_base_moomoo.loc[inv_yrly_base_moomoo.index == i, "Open_mkt_value"] = round(
        sum(open_stock_mkt_value_moomoo_list), 2
    )
    print(f"  Year {i} total open mkt value: {round(sum(open_stock_mkt_value_moomoo_list),2)}\n")


2024 ['CE' 'KR' 'MSFT' 'SGOV' 'SOWG']
  CE: qty=42.0, price=69.21, value=2906.82
  KR: qty=50.0, price=61.15, value=3057.50
  MSFT: qty=7.0, price=421.50, value=2950.50
  SGOV: qty=100.0, price=100.32, value=10032.00
  SOWG: qty=716.0, price=30.58, value=21898.65
  Year 2024 total open mkt value: 40845.47

2025 ['ACN' 'CL' 'LPG' 'META' 'NVDA' 'V']
  ACN: qty=64.0, price=268.30, value=17171.20
  CL: qty=87.0, price=79.02, value=6874.74
  LPG: qty=10.0, price=24.34, value=243.40
  META: qty=14.0, price=660.09, value=9241.26
  NVDA: qty=22.0, price=186.50, value=4103.00
  V: qty=25.0, price=350.71, value=8767.75
  Year 2025 total open mkt value: 46401.35

2026 ['ACN' 'INTU' 'META' 'NVDA' 'V']
  ACN: qty=64.0, price=179.83, value=11509.12
  INTU: qty=5.0, price=399.04, value=1995.20
  META: qty=14.0, price=608.74, value=8522.36
  NVDA: qty=22.0, price=198.45, value=4365.90
  V: qty=36.0, price=328.03, value=11809.08
  Year 2026 total open mkt value: 38201.66



# RBC


In [13]:
inv_yrly_base_rbc = pd.DataFrame({
    "Year":             [2019, 2020, 2021, 2022, 2023, 2024],
    "Deposits":         [0, 10000, 0, 0, 20000, 0],
    "Withdrawals":      [0, 0, 0, 0, 0, -41083.87],
    "Dividends":        [0, 0, 0, 0, 0, 0],
    "Closed_gain/loss": [0, -10000, -10000, -10000, -30000, 41083.87],
    "Open_mkt_value":   [0, 10000, 10000, 10000, 30000, 0],
})
fx_for_merge = fx_CADUSD_yrly_df.reset_index(drop=True)
inv_yrly_base_rbc = inv_yrly_base_rbc.merge(fx_for_merge, on="Year", how="left")
inv_yrly_base_rbc["Closed_gain/loss"] *= inv_yrly_base_rbc["CADUSD"]
inv_yrly_base_rbc["Open_mkt_value"]   *= inv_yrly_base_rbc["CADUSD"]
inv_yrly_base_rbc.drop(columns=["CADUSD"], inplace=True)
inv_yrly_base_rbc.set_index("Year", inplace=True)


# Consolidation


In [14]:
inv_yrly_base       = inv_yrly_base.reset_index(drop=False)
inv_yrly_base_moomoo = inv_yrly_base_moomoo.reset_index(drop=False)
# inv_yrly_base_rbc = inv_yrly_base_rbc.reset_index(drop=False)  # RBC excluded from merge

inv_yrly_base_merge = (
    pd.concat([inv_yrly_base, inv_yrly_base_moomoo])
    .groupby("Year").sum().reset_index(drop=False)
    .fillna(0)
)

# Yearly FX reference column
CADUSD_mapping = dict(zip(fx_CADUSD_yrly_df["Year"], fx_CADUSD_yrly_df["CADUSD"]))
inv_yrly_base_merge["CADUSD_forx"] = inv_yrly_base_merge["Year"].map(CADUSD_mapping)

# Merge all brokerage trades
trades_df_merge = pd.concat([trades_df_questrade, trades_df_moomoo], ignore_index=True)


In [15]:
# ── Cumulative columns ──────────────────────────────────────────────────────
inv_yrly_base_merge["Activity_Yr"]         = inv_yrly_base_merge["Year"]
inv_yrly_base_merge["Principal"]           = inv_yrly_base_merge["Deposits"] + inv_yrly_base_merge["Withdrawals"]
inv_yrly_base_merge["Principal_after_div"] = inv_yrly_base_merge["Principal"] + inv_yrly_base_merge["Dividends"]

inv_yrly_base_merge["Running_Principal"]           = inv_yrly_base_merge["Principal"].cumsum()
inv_yrly_base_merge["Running_Dividends"]           = inv_yrly_base_merge["Dividends"].cumsum()
inv_yrly_base_merge["Running_Closed_gain/loss"]    = inv_yrly_base_merge["Closed_gain/loss"].cumsum()
inv_yrly_base_merge["Running_Principal_after_div"] = inv_yrly_base_merge["Principal_after_div"].cumsum()

inv_yrly_base_merge["Year_end_capital_value"] = (
    inv_yrly_base_merge["Running_Principal_after_div"]
    + inv_yrly_base_merge["Running_Closed_gain/loss"]
    + inv_yrly_base_merge["Open_mkt_value"]
)

# ── Realized P&L via avg-cost ledger (reference) ────────────────────────────
_t = trades_df_merge.copy()
_t["_sort_key"] = _t["Action"].map({"Buy": 0, "Sell": 1})
_t = _t.sort_values(["Transaction_Date", "_sort_key"]).reset_index(drop=True)
realized_pnl_by_year = {int(y): 0.0 for y in inv_yrly_base_merge["Year"].unique()}
_inventory = {}
for _, row in _t.iterrows():
    sym = row["Symbol"]; yr = int(row["Year"])
    if sym not in _inventory:
        _inventory[sym] = {"qty": 0.0, "cost": 0.0}
    if row["Action"] == "Buy":
        _inventory[sym]["qty"]  += row["Quantity"]
        _inventory[sym]["cost"] += -row["Net_Amount"]
    elif row["Action"] == "Sell":
        sell_qty = abs(row["Quantity"])
        if _inventory[sym]["qty"] > 0:
            avg_cost = _inventory[sym]["cost"] / _inventory[sym]["qty"]
            realized_pnl_by_year[yr] = realized_pnl_by_year.get(yr, 0.0) + (row["Net_Amount"] - avg_cost * sell_qty)
            _inventory[sym]["qty"]  -= sell_qty
            _inventory[sym]["cost"] -= avg_cost * sell_qty
inv_yrly_base_merge["Realized_PnL"] = inv_yrly_base_merge["Year"].map(realized_pnl_by_year).fillna(0)

# ── Cash flow series ────────────────────────────────────────────────────────
cash_flow_df = pd.concat([questrade_cf_df, moomoo_cf_df], ignore_index=True)
cash_flow_df["Transaction_Date"] = pd.to_datetime(cash_flow_df["Transaction_Date"])
cash_flow_df = cash_flow_df.sort_values("Transaction_Date").reset_index(drop=True)
today = pd.Timestamp.today().normalize()

inv_yrly_base_merge = inv_yrly_base_merge.sort_values("Year").reset_index(drop=True)
inv_yrly_base_merge["Year_start_capital_value"] = (
    inv_yrly_base_merge["Year_end_capital_value"].shift(1).fillna(0)
)

# ── Modified Dietz TWR (GIPS-compliant) ─────────────────────────────────────
def modified_dietz_twr(year, v_start_mkt, v_end_mkt, cf_df_all, today):
    year_start  = pd.Timestamp(f"{int(year)}-01-01")
    period_end  = min(pd.Timestamp(f"{int(year)}-12-31"), today)
    period_days = (period_end - year_start).days
    if period_days <= 0:
        return float("nan")
    in_yr = cf_df_all[
        (cf_df_all["Transaction_Date"] >= year_start)
        & (cf_df_all["Transaction_Date"] <= period_end)
    ]
    if in_yr.empty:
        net_cf = 0.0; weighted_cf = 0.0
    else:
        net_cf      = float(in_yr["Net_Amount"].sum())
        days_after  = (period_end - in_yr["Transaction_Date"]).dt.days
        weighted_cf = float((days_after / period_days * in_yr["Net_Amount"]).sum())
    numer = v_end_mkt - v_start_mkt - net_cf
    denom = v_start_mkt + weighted_cf
    return numer / denom if denom != 0 else float("nan")

inv_yrly_base_merge["Ratio_Yearly_Return_%"] = [
    modified_dietz_twr(
        row["Year"], row["Year_start_capital_value"],
        row["Year_end_capital_value"], cash_flow_df, today,
    ) * 100
    for _, row in inv_yrly_base_merge.iterrows()
]
inv_yrly_base_merge["Ratio_Dividends_Yield_%"] = (
    inv_yrly_base_merge["Dividends"] / inv_yrly_base_merge["Running_Principal_after_div"]
) * 100

# ── CAGR (chain-linked TWR) ──────────────────────────────────────────────────
start_date = pd.Timestamp(f"{int(inv_yrly_base_merge['Year'].min())}-01-01")
n_years    = (today - start_date).days / 365.25
HPR_portfolio = (1 + inv_yrly_base_merge["Ratio_Yearly_Return_%"] / 100).prod()
inv_yrly_base_merge["CAGR_%"] = (HPR_portfolio ** (1 / n_years) - 1) * 100

CAGR_ending_value_sp500   = sp500_hist_annual[sp500_hist_annual["Year"] == today.year]["stock_price"].values.round(2)[0]
CAGR_starting_value_sp500 = sp500_hist_annual[sp500_hist_annual["Year"] == int(inv_yrly_base_merge["Year"].min()) - 1]["stock_price"].values.round(2)[0]
inv_yrly_base_merge["sp500_CAGR_%"] = (
    (CAGR_ending_value_sp500 / CAGR_starting_value_sp500) ** (1 / n_years) - 1
) * 100

# ── XIRR / MWR ──────────────────────────────────────────────────────────────
def xirr(dates, amounts, lo=-0.99, hi=10.0, tol=1e-9, max_iter=200):
    if not dates or len(dates) != len(amounts):
        return None
    pairs = sorted(zip(dates, amounts), key=lambda p: p[0])
    dates_s, amounts_s = zip(*pairs)
    t0 = dates_s[0]; days = [(d - t0).days for d in dates_s]
    def npv(r): return sum(a / (1 + r) ** (d / 365.25) for d, a in zip(days, amounts_s))
    f_lo, f_hi = npv(lo), npv(hi)
    if f_lo * f_hi > 0:
        return None
    for _ in range(max_iter):
        mid = 0.5 * (lo + hi); f_mid = npv(mid)
        if abs(f_mid) < tol: return mid
        if f_lo * f_mid < 0: hi, f_hi = mid, f_mid
        else: lo, f_lo = mid, f_mid
    return 0.5 * (lo + hi)

xirr_dates   = list(cash_flow_df["Transaction_Date"])
xirr_amounts = [-x for x in cash_flow_df["Net_Amount"].tolist()]
xirr_dates.append(today)
xirr_amounts.append(inv_yrly_base_merge["Year_end_capital_value"].iloc[-1])
irr_value = xirr(xirr_dates, xirr_amounts)
inv_yrly_base_merge["MWR_IRR_%"] = (irr_value * 100) if irr_value is not None else float("nan")
inv_yrly_base_merge.reset_index(drop=True, inplace=True)

# ── Merge with S&P 500 annual returns ────────────────────────────────────────
inv_yrly_merged = inv_yrly_base_merge.merge(
    sp500_hist_annual, how="left", left_on="Activity_Yr", right_on="Year"
)
inv_yrly_merged.rename(columns={"Annual Return": "sp500_without_div_%"}, inplace=True)

# ── Final consolidate table ───────────────────────────────────────────────────
_display_cols = ["Activity_Yr", "Ratio_Yearly_Return_%", "sp500_without_div_%",
                 "CAGR_%", "sp500_CAGR_%", "MWR_IRR_%"]
_round_cols   = ["Ratio_Yearly_Return_%", "sp500_without_div_%", "CAGR_%", "sp500_CAGR_%", "MWR_IRR_%"]
inv_yrly_merged[_round_cols] = inv_yrly_merged[_round_cols].round(2)
inv_yrly_merged_consolidate  = inv_yrly_merged[_display_cols]


# Portfolio Reports


In [16]:
inv_yrly_merged_consolidate


,Activity_Yr,Ratio_Yearly_Return_%,sp500_without_div_%,CAGR_%,sp500_CAGR_%,MWR_IRR_%
0,2019,24.08,28.79,7.28,15.53,5.6
1,2020,-38.13,16.16,7.28,15.53,5.6
2,2021,78.35,27.04,7.28,15.53,5.6
3,2022,-44.58,-19.48,7.28,15.53,5.6
4,2023,68.00,24.29,7.28,15.53,5.6
5,2024,37.36,23.30,7.28,15.53,5.6
6,2025,-4.28,16.35,7.28,15.53,5.6
7,2026,-0.10,5.68,7.28,15.53,5.6


In [17]:
open_buy_df_previous = pd.DataFrame()
year_end_open_stock_dict = {}

for i in sorted(trades_df_merge["Year"].unique()):
    year_end_open_stock_dict[i] = {"Symbol": [], "Quantity": []}
    yearly_trades_df = pd.concat(
        [trades_df_merge[trades_df_merge["Year"] == i], open_buy_df_previous], ignore_index=True
    )
    buy_df  = yearly_trades_df[yearly_trades_df["Action"] == "Buy"]
    sell_df = yearly_trades_df[yearly_trades_df["Action"] == "Sell"]

    buy_grp  = buy_df.groupby("Symbol")["Quantity"].sum()
    sell_grp = sell_df.groupby("Symbol")["Quantity"].sum().reindex(buy_grp.index, fill_value=0)
    open_grp_df = (buy_grp + sell_grp)[(buy_grp + sell_grp) > 0].reset_index()

    open_buy_df = pd.merge(open_grp_df, buy_df, on="Symbol", how="left")
    open_buy_df = open_buy_df.drop_duplicates(subset=["Symbol", "Quantity_x"], keep="first")
    open_buy_df = open_buy_df[[
        "Transaction_Date", "Action", "Symbol", "Quantity_x", "Price", "Net_Amount", "Year"
    ]].rename(columns={"Quantity_x": "Quantity"})
    open_buy_df_previous = open_buy_df.copy()

    print(i, open_buy_df["Symbol"].unique())
    for symbol in open_buy_df["Symbol"].unique():
        qty = open_buy_df[open_buy_df["Symbol"] == symbol]["Quantity"].values.sum()
        year_end_open_stock_dict[i]["Symbol"].append(symbol)
        year_end_open_stock_dict[i]["Quantity"].append(qty)

# Current holdings transaction history
current_yr_open_stock_list = year_end_open_stock_dict[datetime.today().year]["Symbol"]
current_open_holding_df = pd.concat([
    trades_df_merge[trades_df_merge["Symbol"] == sym]
    for sym in set(current_yr_open_stock_list)
]).drop_duplicates().sort_values("Transaction_Date", ascending=True)


2019 ['BABA' 'DDOG' 'ORCL']
2020 ['AAL' 'AMZN' 'DDOG']
2021 ['FDX' 'META' 'TSM']
2022 ['FDX' 'META' 'TSM']
2023 ['TSM']
2024 ['CE' 'GOOG' 'KR' 'MSFT' 'SGOV' 'SOWG' 'TGT' 'TSM']
2025 ['ACN' 'CL' 'CMR.TO' 'GOOG' 'LPG' 'META' 'NVDA' 'SGOV' 'TGT' 'TSM' 'V']
2026 ['ACN' 'CMR.TO' 'GOOG' 'INTU' 'META' 'MSFT' 'NVDA' 'SKWD' 'TGT' 'TSM' 'V']


In [18]:
current_open_holding_cost_dict = {}
for symbol in current_open_holding_df["Symbol"].unique():
    df = current_open_holding_df[current_open_holding_df["Symbol"] == symbol].copy()
    df["Rolling_Capital"]         = df["Net_Amount"].cumsum()
    df["Rolling_Quantity"]        = df["Quantity"].cumsum()
    df["Rolling_Price_Per_Share"] = df["Rolling_Capital"] / df["Rolling_Quantity"]
    current_open_holding_cost_dict[symbol] = df[[
        "Transaction_Date", "Symbol", "Action", "Quantity",
        "Rolling_Quantity", "Price", "Rolling_Capital", "Rolling_Price_Per_Share",
    ]]

consolidated_holding_cost_df = pd.concat(current_open_holding_cost_dict.values())
dupe_groups = consolidated_holding_cost_df.groupby(["Symbol", "Transaction_Date"]).cumcount()
consolidated_holding_cost_df["Transaction_Date"] += pd.to_timedelta(dupe_groups, unit="d")

consolidated_holding_cost_df_latest = (
    consolidated_holding_cost_df
    .loc[consolidated_holding_cost_df.groupby("Symbol")["Transaction_Date"].idxmax()]
    .reset_index(drop=True)
)
consolidated_holding_cost_df_latest["position_%"] = np.nan


In [19]:
for symbol in consolidated_holding_cost_df_latest["Symbol"].unique():
    Daily_stock_df, _ = fetch_av_daily_split_adjusted(symbol, alpha_vantage_api_key)
    latest_price = Daily_stock_df["stock_price"].iloc[0]
    consolidated_holding_cost_df_latest.loc[
        consolidated_holding_cost_df_latest["Symbol"] == symbol, "latest_stock_price"
    ] = latest_price

    rolling_capital = consolidated_holding_cost_df_latest.loc[
        consolidated_holding_cost_df_latest["Symbol"] == symbol, "Rolling_Capital"
    ].values
    # Cost-based position% (intentional; see inv_summary_01 for rationale)
    holding_cost = abs(rolling_capital) if rolling_capital < 0 else 1
    consolidated_holding_cost_df_latest.loc[
        consolidated_holding_cost_df_latest["Symbol"] == symbol, "position_%"
    ] = ((holding_cost / inv_yrly_merged["Year_end_capital_value"].values[-1]) * 100).round(2)
    consolidated_holding_cost_df_latest.loc[
        consolidated_holding_cost_df_latest["Symbol"] == symbol, "Year_end_capital_value"
    ] = inv_yrly_merged["Year_end_capital_value"].values[-1]

# Next earnings date
consolidated_holding_cost_df_latest["Next_qtr_date"] = None
for symbol in consolidated_holding_cost_df_latest["Symbol"].unique():
    CSV_URL = (f"https://www.alphavantage.co/query?function=EARNINGS_CALENDAR"
               f"&symbol={symbol}&horizon=12month&apikey={alpha_vantage_api_key}")
    with requests.Session() as s:
        decoded = s.get(CSV_URL, timeout=30).content.decode("utf-8")
        rows = list(csv.reader(decoded.splitlines(), delimiter=","))
        forecast_df = pd.DataFrame(columns=rows[0], data=rows[1:])
        if "reportDate" in forecast_df.columns and not forecast_df.empty:
            consolidated_holding_cost_df_latest.loc[
                consolidated_holding_cost_df_latest["Symbol"] == symbol, "Next_qtr_date"
            ] = forecast_df["reportDate"].head(1).values[0]

consolidated_holding_cost_df_latest = consolidated_holding_cost_df_latest[[
    "Transaction_Date", "Symbol", "Rolling_Quantity", "Rolling_Capital",
    "Rolling_Price_Per_Share", "position_%", "Year_end_capital_value",
    "latest_stock_price", "Next_qtr_date",
]]


In [20]:
consolidated_holding_cost_df_latest.sort_values(by='position_%', ascending=False)


,Transaction_Date,Symbol,Rolling_Quantity,Rolling_Capital,Rolling_Price_Per_Share,position_%,Year_end_capital_value,latest_stock_price,Next_qtr_date
1,2026-02-05,CMR.TO,1398.0,-50624.827802,-36.212323,26.78,189034.572293,50.03,None
0,2026-04-17,ACN,119.0,-27355.091737,-229.874720,14.47,189034.572293,179.83,2026-06-18
5,2026-04-25,MSFT,63.0,-23880.338495,-379.052992,12.63,189034.572293,414.44,None
7,2026-02-25,SKWD,488.0,-22492.290000,-46.090758,11.90,189034.572293,45.44,2026-05-06
3,2026-04-17,INTU,46.0,-17655.378000,-383.812565,9.34,189034.572293,399.04,f
10,2026-04-15,V,45.0,-15252.905100,-338.953447,8.07,189034.572293,328.03,f
6,2025-11-26,NVDA,50.0,-9405.840000,-188.116800,4.98,189034.572293,198.45,2026-05-20
4,2025-11-10,META,14.0,-8851.900000,-632.278571,4.68,189034.572293,608.74,None
8,2024-07-17,TGT,22.0,-3438.050000,-156.275000,1.82,189034.572293,128.89,2026-05-20
2,2025-09-12,GOOG,10.0,7293.286100,729.328610,0.00,189034.572293,383.22,None


In [21]:
port_holding_pct = consolidated_holding_cost_df_latest["position_%"].sum()
port_cash_pct    = 100 - port_holding_pct
print(f"Portfolio holding position: {port_holding_pct:.2f}")
print(f"Cash position:              {port_cash_pct:.2f}")


Portfolio holding position: 94.67
Cash position:              5.33
